In [1]:
import numpy as np
import pandas as pd
import pathlib
import matplotlib.pyplot as plt
from neurodsp.spectral import compute_spectrum
from neurodsp.plts.spectral import plot_power_spectra
import os
from joblib import Parallel, delayed
from tqdm import tqdm

In [2]:
# define file paths
eeg_path = '/oscar/data/sjones/shared/TDBRAIN_preprocessed/preprocessed'
metadata_path = '/oscar/home/scbao/tdbrain-model/data/TDBRAIN_participants_V2.tsv'
subj_list = os.listdir(eeg_path)
print("Number of total subjects:", len(subj_list))

# read data
metadata_df = pd.read_csv(metadata_path, delimiter='\t')

Number of total subjects: 1274


In [3]:
# generate mask to pull out MDD, DISC subjects only
subj_mask = np.isin(metadata_df['participants_ID'].values, subj_list)
discovery_mask = metadata_df['DISC/REP'].values == 'DISCOVERY'
dataset_mask = metadata_df['Dataset'].values == 'MDD-rTMS'
rTMS_mask = ~metadata_df['rTMS PROTOCOL'].isna() # excludes 1 participant

mask = np.logical_and.reduce([subj_mask, discovery_mask, dataset_mask, rTMS_mask])

df = metadata_df[mask].copy() # .copy because plan to add columns later
print("Shape after keeping MDD-rTMS, Discovery rows:", df.shape)

Shape after keeping MDD-rTMS, Discovery rows: (131, 111)


In [4]:
# save paths to eeg numpy files for each subject in df
ec_eeg_path, eo_eeg_path, has_ses1 = list(), list(), list()

for subj_id in df['participants_ID'].values:
    subj_path = f'{eeg_path}/{subj_id}/ses-1/eeg'
    if os.path.isdir(subj_path):
        has_ses1.append(True)
        subj_files = os.listdir(subj_path)
        ec_subj_files = list(pathlib.Path(subj_path).glob('*restEC*.npy'))
        eo_subj_files = list(pathlib.Path(subj_path).glob('*restEO*.npy'))
        assert len(ec_subj_files) == len(eo_subj_files) == 1

        ec_eeg_path.append(str(ec_subj_files[0]))
        eo_eeg_path.append(str(eo_subj_files[0]))

    else:
        has_ses1.append(False)
        ec_eeg_path.append('')
        eo_eeg_path.append('')

df['ec_eeg_path'] = ec_eeg_path
df['eo_eeg_path'] = eo_eeg_path
df['has_ses1'] = has_ses1

df = df[df['has_ses1'] == True].reset_index(drop=True)
print("Shape after removing subj with no session 1 data:", df.shape)

Shape after removing subj with no session 1 data: (128, 114)


In [5]:
# define feature extraction function
def get_bandpower_features(subj_data_path, condition, normalize=True):
    channel_filter = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC3', 'FCz', 'FC4',
                      'T7', 'C3', 'Cz', 'C4', 'T8', 'CP3', 'CPz', 'CP4', 'P7', 'P3',
                      'Pz', 'P4', 'P8', 'O1', 'Oz', 'O2',]

    eeg_dict = np.load(subj_data_path, allow_pickle=True)
    channel_labels = eeg_dict['labels']
    fs = eeg_dict['Fs']

    channel_mask = np.isin(channel_labels, channel_filter)
    eeg_data = eeg_dict['data'][0, channel_mask, :]
    # print(eeg_data.shape)
    assert eeg_data.shape[0] == len(channel_filter)

    bands = {
        'delta': [0.5, 4], 
        'theta': [4, 8], 
        'alpha': [8, 12], 
        'low_beta': [12, 20],
        'high_beta': [20, 30], 
        'low_gamma': [30, 40], 
        'high_gamma': [40, 80]
    }

    # construct dict with bandpower values for inputted subj
    feature_dict = {}

    for ch_idx, ch_name in enumerate(channel_filter):
        freqs, psd = compute_spectrum(eeg_data[ch_idx], fs, method='welch', avg_type='mean', nperseg=fs*2)
        
        band_powers = {}
        for band_name, (fmin, fmax) in bands.items():
            idx = (freqs >= fmin) & (freqs <= fmax)
            band_power = np.trapezoid(psd[idx], freqs[idx])
            band_powers[band_name] = band_power

        # normalize across all band powers for each channel
        if normalize:
            total_power = sum(band_powers.values()) + 1e-8
            for band_name in band_powers:
                band_powers[band_name] /= total_power

        for band_name, value in band_powers.items():
            feature_dict[f'{condition}_{ch_name}_{band_name}_power'] = value

    return feature_dict

def extract_subj_features(subj_id, ec_path, eo_path):
    feats = {'participants_ID': subj_id}
    feats.update(get_bandpower_features(ec_path, 'EC'))
    feats.update(get_bandpower_features(eo_path, 'EO'))
    return feats

In [6]:
# extract features in parallel for all subjects
res = Parallel(n_jobs=16)(
    delayed(extract_subj_features)(subj_id, ec_path, eo_path)
    for subj_id, ec_path, eo_path in tqdm(
        zip(df['participants_ID'].values,
            df['ec_eeg_path'].values,
            df['eo_eeg_path'].values),
        total=len(df)
    )
)

100%|██████████| 128/128 [00:03<00:00, 40.06it/s]


In [7]:
# save results in pickle file
bandpower_df = pd.DataFrame(res)
bandpower_df.to_pickle(f'../data/mdd_rtms_bandpower_all.pkl')
print("Shape of bandpower dataframe:", bandpower_df.shape)
print(bandpower_df.head())

Shape of bandpower dataframe: (128, 365)
  participants_ID  EC_Fp1_delta_power  EC_Fp1_theta_power  EC_Fp1_alpha_power  \
0    sub-87999321            0.440963            0.125691            0.169726   
1    sub-88000181            0.213105            0.201495            0.197796   
2    sub-88000313            0.198156            0.108387            0.435276   
3    sub-88000489            0.210357            0.230957            0.329855   
4    sub-88000533            0.299961            0.092930            0.089858   

   EC_Fp1_low_beta_power  EC_Fp1_high_beta_power  EC_Fp1_low_gamma_power  \
0               0.048599                0.053521                0.037638   
1               0.158821                0.061693                0.041036   
2               0.153407                0.047434                0.019371   
3               0.059994                0.055549                0.024703   
4               0.080159                0.118211                0.087667   

   EC_Fp1_high_